In [105]:
library(jsonlite)
library(dplyr)
library(rstatix)
library(purrr)
library(ggplot2)
library(tibble)

In [106]:
output_dir  <- "./output/finished"

all_data    <- stream_in(file(paste(output_dir, "/all_results.jsonl", sep="")), verbose = FALSE) %>%
mutate(
  pass_duration = ifelse(test_passed, eval_duration, NA)
)
agg_data    <- stream_in(file(paste(output_dir, "/agg_results.jsonl", sep="")), verbose = FALSE) %>%
  mutate(pass_rate = tests_passed / executions) %>%
  relocate(pass_rate, .after = tests_passed)

In [107]:
head(all_data)
#str(all_data)
table(all_data$dataset)

head(agg_data)
#str(agg_data)
table(agg_data$dataset)

,dataset,task_id,prompt_id,execution_id,test_passed,error,traceback,response_len,gen_duration,eval_duration,pass_duration
,<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
1,json,HumanEval/0,0,0,TRUE,NA,NA,251,1.4114334,0.0002278,0.0002278
2,json,HumanEval/0,0,1,TRUE,NA,NA,251,0.9672739,0.0001935,0.0001935
3,json,HumanEval/0,0,2,TRUE,NA,NA,246,0.7820504,0.0001729,0.0001729
4,json,HumanEval/0,0,3,TRUE,NA,NA,246,1.1949659,0.0002066,0.0002066
5,json,HumanEval/0,0,4,TRUE,NA,NA,246,0.9941327,0.0001844,0.0001844
6,json,HumanEval/0,0,5,TRUE,NA,NA,246,0.9450328,0.0001907,0.0001907



    json markdown    tuned  vanilla     yaml 
    1640     1640     1640     1640     1640 

,dataset,task_id,prompt_id,executions,tests_passed,pass_rate,response_len_mean,response_len_std,intra_prompt_rougeL,gen_duration_mean,gen_duration_std,eval_duration_mean,eval_duration_std,pass_duration_mean,pass_duration_std
,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,json,HumanEval/0,0,10,10,1.0,251.9,11.282287,0.9634564,1.2431588,0.5606541,0.00022192,3.468374e-05,0.0002219200,3.468374e-05
2,json,HumanEval/1,1,10,10,1.0,509.1,24.809071,0.7459778,2.3676615,0.7951114,0.00017785,3.554342e-05,0.0001778500,3.554342e-05
3,json,HumanEval/10,10,10,10,1.0,297.8,26.141155,0.8395449,1.7490450,0.2798334,0.00014355,1.685659e-05,0.0001435500,1.685659e-05
4,json,HumanEval/100,100,10,10,1.0,148.8,31.783014,0.7016743,1.3428578,0.2444828,0.00011586,1.361508e-05,0.0001158600,1.361508e-05
5,json,HumanEval/101,101,10,5,0.5,76.4,5.817216,0.7177749,0.7959978,0.2981849,0.00017585,6.548037e-05,0.0001257600,2.191325e-05
6,json,HumanEval/102,102,10,9,0.9,170.3,27.107379,0.7633642,1.2602217,0.2034964,0.00011739,3.347403e-05,0.0001065111,7.842822e-06



    json markdown    tuned  vanilla     yaml 
     164      164      164      164      164 

### Aggregate per Dataset Type

In [117]:
all_summary <- all_data %>%
  group_by(dataset) %>%
  summarise(
    # Globally over all prompts
    response_len_global_median = median(response_len),
    inter_prompt_response_len_std = sd(response_len),
    
    gen_duration_global_median = median(gen_duration),
    inter_prompt_gen_duration_std = sd(gen_duration),
    
    eval_duration_global_median = median(eval_duration),
    inter_prompt_eval_duration_std = sd(eval_duration),
    
    pass_duration_global_median = median(pass_duration, na.rm = TRUE),
    inter_prompt_pass_duration_std = sd(pass_duration, na.rm = TRUE)
  )
#head(all_summary)

agg_summary <- agg_data %>%
  group_by(dataset) %>%
  summarise(
    tests_passed = sum(tests_passed),
    pass_rate = sum(tests_passed)/sum(executions),
    
    intra_prompt_rougeL_median = median(intra_prompt_rougeL),
    intra_prompt_rougeL_mean = mean(intra_prompt_rougeL),
    intra_prompt_rougeL_std = sd(intra_prompt_rougeL),
    
    # Per prompt (mean means are equivalent because #executions is consistent)
    response_len_median = median(response_len_mean),
    response_len_mean = mean(response_len_mean),
    intra_prompt_response_len_std = mean(response_len_std),
    
    gen_duration_median = median(gen_duration_mean),
    gen_duration_mean = mean(gen_duration_mean),
    intra_prompt_gen_duration_std = mean(gen_duration_std),
    
    eval_duration_median = median(eval_duration_mean),
    eval_duration_mean = mean(eval_duration_mean),
    intra_prompt_eval_duration_std = mean(eval_duration_std),
    
    pass_duration_median = median(pass_duration_mean, na.rm = TRUE),
    pass_duration_mean = mean(pass_duration_mean, na.rm = TRUE),
    intra_prompt_pass_duration_std = mean(pass_duration_std, na.rm = TRUE)
  )
#head(agg_summary)

summary <- left_join(agg_summary, all_summary, by = c("dataset")) %>%
  relocate(inter_prompt_response_len_std, .after = intra_prompt_response_len_std) %>%
  relocate(inter_prompt_gen_duration_std, .after = intra_prompt_gen_duration_std) %>%
  relocate(inter_prompt_eval_duration_std, .after = intra_prompt_eval_duration_std) %>%
  relocate(inter_prompt_pass_duration_std, .after = intra_prompt_pass_duration_std) %>%
  
  relocate(response_len_global_median, .after = response_len_median) %>%
  relocate(gen_duration_global_median, .after = gen_duration_median) %>%
  relocate(eval_duration_global_median, .after = eval_duration_median) %>%
  relocate(pass_duration_global_median, .after = pass_duration_median)

summary$dataset <- factor(summary$dataset, levels = c("vanilla", "json", "markdown", "yaml", "tuned"))

head(summary)

dataset,tests_passed,pass_rate,intra_prompt_rougeL_median,intra_prompt_rougeL_mean,intra_prompt_rougeL_std,response_len_median,response_len_global_median,response_len_mean,intra_prompt_response_len_std,⋯,eval_duration_median,eval_duration_global_median,eval_duration_mean,intra_prompt_eval_duration_std,inter_prompt_eval_duration_std,pass_duration_median,pass_duration_global_median,pass_duration_mean,intra_prompt_pass_duration_std,inter_prompt_pass_duration_std
<fct>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
vanilla,1453,0.8859756,0.8178819,0.7870130,0.1521156,170.80,169.5,209.4183,44.44705,⋯,0.000156935,0.00015245,0.0289616181,0.0382819758,0.609734320,0.0001456250,0.00014200,0.0746764351,0.0244935868,0.647716324
tuned,1226,0.7475610,0.8195558,0.7976288,0.1616137,178.70,172.0,212.0323,37.77243,⋯,0.000162405,0.00015745,0.0002857741,0.0001687185,0.001363804,0.0001410225,0.00013690,0.0003532783,0.0001780874,0.001576317
json,1479,0.9018293,0.8589636,0.8418869,0.1364649,183.10,176.5,213.4573,27.88412,⋯,0.000144370,0.00014455,0.0285506785,0.0102248072,0.383255549,0.0001394043,0.00013720,0.0300085747,0.0107299619,0.403470480
markdown,1460,0.8902439,0.8604895,0.8209732,0.1483368,200.60,189.5,228.1360,33.39333,⋯,0.000150050,0.00014930,1.1554968196,3.4413865142,46.165378141,0.0001446900,0.00014195,0.0232450131,0.0238769732,0.355781454
yaml,1432,0.8731707,0.8610281,0.8369692,0.1382367,179.05,173.0,214.9360,28.48774,⋯,0.000159275,0.00015380,0.0146193516,0.0179510703,0.287811989,0.0001494500,0.00014540,0.0304697547,0.0157511642,0.307964139


# Dependent Variables

### Check for Normality ⇒ All in Violation

In [109]:
# Apply Shapiro-Wilk test for each dataset group
agg_data %>%
  group_by(dataset) %>%
  summarise(
    tests_passed_p_value = shapiro.test(tests_passed)$p.value,
    pass_rate_p_value = shapiro.test(pass_rate)$p.value,
    pass_rate_normal = pass_rate_p_value > 0.05,

    intra_prompt_rougeL_p_value = shapiro.test(intra_prompt_rougeL)$p.value,
    intra_prompt_rougeL_normal = intra_prompt_rougeL_p_value > 0.05,

    response_len_mean_p_value = shapiro.test(response_len_mean)$p.value,
    response_len_mean_normal = response_len_mean_p_value > 0.05,

    gen_duration_mean_p_value = shapiro.test(gen_duration_mean)$p.value,
    gen_duration_mean_normal = gen_duration_mean_p_value > 0.05,

    eval_duration_mean_p_value = shapiro.test(eval_duration_mean)$p.value,
    eval_duration_mean_normal = eval_duration_mean_p_value > 0.05,

    pass_duration_mean_p_value = shapiro.test(pass_duration_mean)$p.value,
    pass_duration_mean_normal = pass_duration_mean_p_value > 0.05
  )

dataset,tests_passed_p_value,pass_rate_p_value,pass_rate_normal,intra_prompt_rougeL_p_value,intra_prompt_rougeL_normal,response_len_mean_p_value,response_len_mean_normal,gen_duration_mean_p_value,gen_duration_mean_normal,eval_duration_mean_p_value,eval_duration_mean_normal,pass_duration_mean_p_value,pass_duration_mean_normal
<chr>,<dbl>,<dbl>,<lgl>,<dbl>,<lgl>,<dbl>,<lgl>,<dbl>,<lgl>,<dbl>,<lgl>,<dbl>,<lgl>
json,1.182736e-22,1.182736e-22,FALSE,8.864788e-08,FALSE,8.621955e-11,FALSE,6.022329e-21,FALSE,6.638408e-28,FALSE,2.770545e-27,FALSE
markdown,4.683062e-22,4.683062e-22,FALSE,2.554627e-07,FALSE,3.267383e-10,FALSE,2.541912e-11,FALSE,6.766472e-28,FALSE,2.797122e-27,FALSE
tuned,5.982803e-19,5.982803e-19,FALSE,7.642192e-07,FALSE,5.439119e-11,FALSE,4.967964e-12,FALSE,4.716225e-26,FALSE,8.490915e-24,FALSE
vanilla,4.599736e-21,4.599736e-21,FALSE,4.138176e-05,FALSE,5.228319e-11,FALSE,9.544795e-10,FALSE,6.648837e-28,FALSE,1.918785e-27,FALSE
yaml,2.986801e-21,2.986801e-21,FALSE,1.411295e-07,FALSE,2.081826e-10,FALSE,4.958737e-09,FALSE,6.849767e-28,FALSE,3.360029e-27,FALSE


### ⇒ Conduct Non-Parametric Tests

In [110]:
dependent_vars <- c("pass_rate", "intra_prompt_rougeL", "response_len_mean", "gen_duration_mean", "eval_duration_mean", "pass_duration_mean")

In [111]:
for (var in dependent_vars) {
    # Non-parametric Kruskal-Wallis Test (because the values of all DVs are non-parametric as per Shapiro-Wilk)
    formula <- reformulate("dataset", response = var)
    cat("### Kruskal-Wallis Test for", var, "###\n")
    kruskal <- kruskal.test(formula, data = agg_data)
    print(kruskal)

    # Calculate Effect Sizes (Epsilon-squared)
    H <- kruskal$statistic
    k <- length(unique(agg_data$dataset))
    n <- nrow(agg_data)

    cat(paste("(", H, " - ", k, " + 1) / (", n, " - ", k, ")\n", sep=""))

    epsilon_sq <- as.numeric((H - k + 1) / (n - k))
    # Classify based on Cohen's suggestions
    classification <- "negligible"
    if (epsilon_sq > 0.1379) {
      classification <- "large"
    } else if (epsilon_sq > 0.0588) {
      classification <- "medium"
    } else if (epsilon_sq > 0.0099) {
      classification <- "small"
    }
    cat("  ↳ Effect size (ε²): ", round(epsilon_sq, 4), " (", classification, ")\n\n", sep="")

    # Post-hoc Test if Kruskal-Wallis is significant
    if (kruskal$p.value < 0.05) {
      cat("  ↳ Significant (p = ", kruskal$p.value, ") — Running post-hoc tests:\n", sep="")
      
      # cat("\n>>> Dunn's Test >>>\n")
      # #library(dunn.test)
      # #dunn.test(agg_data[[var]], agg_data$dataset, method = "bonferroni")

      # dunn_results <- dunn_test(formula, data = agg_data, p.adjust.method = "bonferroni")
      # dv_name <- unique(dunn_results$.y.)

      # # Add medians, diffs, effect sizes
      # dunn_results <- dunn_results %>%
      #   mutate(
      #     # median1 = map_dbl(group1, ~ median(agg_data[agg_data$dataset == .x, dv_name][[1]])),
      #     # median2 = map_dbl(group2, ~ median(agg_data[agg_data$dataset == .x, dv_name][[1]])),
      #     median1 = map2_dbl(group1, group2, ~ {
      #       x <- agg_data %>% filter(dataset == .x) %>% pull(dv_name)
      #       y <- agg_data %>% filter(dataset == .y) %>% pull(dv_name)
      #       median(x, na.rm = TRUE)
      #     }),
      #     median2 = map2_dbl(group2, group1, ~ {
      #       x <- agg_data %>% filter(dataset == .x) %>% pull(dv_name)
      #       y <- agg_data %>% filter(dataset == .y) %>% pull(dv_name)
      #       median(x, na.rm = TRUE)
      #     }),
      #     median_diff = median1 - median2,
      #     r = abs(statistic) / sqrt(n1 + n2),
      #     r_class = case_when(
      #       r >= 0.5 ~ "large",
      #       r >= 0.3 ~ "medium",
      #       r >= 0.1 ~ "small",
      #       TRUE     ~ "negligible"
      #     )
      #   ) %>%
      #   select(group1, group2, p.adj, median1, median2, median_diff, r, r_class) %>%
      #   filter(p.adj < 0.05)

      # print(dunn_results)



      cat("\n>>> Pairwise Mann-Whitney U Tests >>>\n")
      # # Just table with p values
      # print(
      #   pairwise.wilcox.test(
      #     x = agg_data[[var]],
      #     g = agg_data$dataset,
      #     p.adjust.method = "bonferroni"  # or "holm", "BH"
      #   )
      # )
      # cat("\nOr in more detail:\n")

      # Also median diff & effect size (r / rank-biserial correlation)
      combos <- combn(unique(agg_data$dataset), 2, simplify = FALSE)
  
      # Collect stats
      results <- data.frame()

      for (pair in combos) {
        group1 <- agg_data %>% filter(dataset == pair[1]) %>% pull(var)
        group2 <- agg_data %>% filter(dataset == pair[2]) %>% pull(var)
        median1 <- median(group1)
        median2 <- median(group2)

        test <- wilcox.test(group1, group2, exact = FALSE, correct = TRUE)

        # Raw p-value
        p_raw <- test$p.value

        # # Z-score approximation
        # z <- qnorm(p_raw / 2, lower.tail = FALSE) * sign(median(group1) - median(group2))
        # r <- z / sqrt(length(group1) + length(group2))
        U <- test$statistic
        n1 <- length(group1)
        n2 <- length(group2)

        # Compute mean and SD of U under H0
        mean_U <- n1 * n2 / 2
        sd_U <- sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

        z <- (U - mean_U) / sd_U
        r <- z / sqrt(n1 + n2)

        # Store result
        results <- rbind(results, data.frame(
          var = var,
          group1 = pair[1],
          group2 = pair[2],
          p_raw = p_raw,
          median1 = median1,
          median2 = median2,
          median_diff = median1 - median2,
          r = r,
          r_class = case_when(
            abs(r) >= 0.5 ~ "large",
            abs(r) >= 0.3 ~ "medium",
            abs(r) >= 0.1 ~ "small",
            TRUE          ~ "negligible"
          )
        ))
      }

      # Adjust p-values
      results$p_adj <- p.adjust(results$p_raw, method = "bonferroni")

      # Print nicely
      results_sig <- results %>% filter(p_adj < 0.05)
      if (nrow(results_sig) == 0) {
        cat("  No significant post-hoc differences after adjustment.\n")
      } else {
        for (i in 1:nrow(results_sig)) {
          row <- results_sig[i, ]
          cat(sprintf("  %s (%.4f) vs. %s (%.4f):\n    adj p = %.4g | median diff = %.4f | r = %.3f (%s)\n",
                      row$group1, row$median1, row$group2, row$median2, row$p_adj, row$median_diff, row$r, row$r_class))
        }
      }
    } else {
      cat("  ↳ Not significant (p = ", kruskal$p.value, ") — Skipping post-hoc.\n", sep="")
    }
    cat("\n\n\n")
}

### Kruskal-Wallis Test for pass_rate ###

	Kruskal-Wallis rank sum test

data:  pass_rate by dataset
Kruskal-Wallis chi-squared = 19.177, df = 4, p-value = 0.0007254

(19.1772350857927 - 5 + 1) / (820 - 5)
  ↳ Effect size (ε²): 0.0186 (small)

  ↳ Significant (p = 0.0007253633) — Running post-hoc tests:

>>> Pairwise Mann-Whitney U Tests >>>
  json (1.0000) vs. tuned (1.0000):
    adj p = 0.002122 | median diff = 0.0000 | r = 0.162 (small)
  markdown (1.0000) vs. tuned (1.0000):
    adj p = 0.006367 | median diff = 0.0000 | r = 0.151 (small)
  tuned (1.0000) vs. vanilla (1.0000):
    adj p = 0.04545 | median diff = 0.0000 | r = -0.128 (small)



### Kruskal-Wallis Test for intra_prompt_rougeL ###

	Kruskal-Wallis rank sum test

data:  intra_prompt_rougeL by dataset
Kruskal-Wallis chi-squared = 14.95, df = 4, p-value = 0.004805

(14.9504703937183 - 5 + 1) / (820 - 5)
  ↳ Effect size (ε²): 0.0134 (small)

  ↳ Significant (p = 0.004805055) — Running post-hoc tests:

>>> Pairwise Mann-Whi

# Plotting

In [112]:
# Reorder the Levels of Dataset
agg_data <- agg_data %>%
  mutate(dataset = factor(dataset, levels = c("vanilla", "tuned", "json", "markdown", "yaml")))

# Define colors per DV level
custom_colors <- c("vanilla" = "#c65154",
                   "tuned" = "#fac05e",
                   "json" = "#89c0c4",
                   "markdown" = "#397aa8",
                   "yaml" = "#163771")

In [114]:
# Convert s to ms for Durations
agg_data <- agg_data %>%
mutate(
    scaled_gen_duration_mean = gen_duration_mean*1000,
    scaled_eval_duration_mean = eval_duration_mean*1000,
    scaled_pass_duration_mean = ifelse(is.na(pass_duration_mean), NA, pass_duration_mean * 1000)
)

### Box Plots

In [ ]:
# Display all data points?
use_jitter <- TRUE

# List of DVs to plot (& settings)
dvs_to_plot <- list(
  # PassRate
  list(col = "pass_rate", label = "Pass Rate", ylab = "Pass Rate", ylim = c(0.4, 0.6), unbound = TRUE, bound = FALSE, print = "unbound"),

  # GenDuration
  list(col = "gen_duration_mean", label = "Generation Duration", ylab = "Generation Time (in s)", ylim = c(0.5, 5.25), unbound = TRUE, bound = TRUE, print = "bound"),

  # EvalDuration
  list(col = "eval_duration_mean", label = "Evaluation Duration", ylab = "Evaluation Time (in s)", ylim = c(0, 5), unbound = TRUE, bound = TRUE, print = "none"),
  list(col = "scaled_eval_duration_mean", label = "Evaluation Duration", ylab = "Evaluation Time (in ms)", ylim = c(0, 17), unbound = FALSE, bound = TRUE, print = "bound"), # half-bound / extended
  #list(col = "scaled_eval_duration_mean", label = "Evaluation Duration", ylab = "Evaluation Time (in ms)", ylim = c(0, 1.25), unbound = FALSE, bound = TRUE, print = "none"),
  list(col = "scaled_eval_duration_mean", label = "Evaluation Duration", ylab = "Evaluation Time (in ms)", ylim = c(0.05, 0.41), unbound = FALSE, bound = TRUE, print = "bound"), # full-bound

  # PassDuration
  #list(col = "pass_duration_mean", label = "Pass Duration", ylab = "Evaluation Time of Passing Responses (in s)", ylim = c(0, 5), unbound = TRUE, bound = TRUE, print = "none"),
  list(col = "scaled_pass_duration_mean", label = "Pass Duration", ylab = "Evaluation Time of Passing Responses (in ms)", ylim = c(0.05, 0.41), unbound = FALSE, bound = TRUE, print = "bound"), # full-bound

  # ROUGE-L
  list(col = "intra_prompt_rougeL", label = "Intra-Prompt ROUGE-L", ylab = "Intra-Prompt ROUGE-L Score", ylim = c(0.4, 1.05), unbound = TRUE, bound = FALSE, print = "unbound"),

  # ResponseLen
  list(col = "response_len_mean", label = "Response Length", ylab = "Response Length (in Number of Characters)", ylim = c(0, 600), unbound = TRUE, bound = FALSE, print = "unbound")
)

# Optional: store plots in a list
plots <- list()

# Loop over DVs
for (dv in dvs_to_plot) {
  # Calculate Number of Data Points for transparent Labeling
  x_labels <- agg_data %>%
    group_by(dataset) %>%
    summarise(n = sum(!is.na(.data[[dv$col]])), .groups = "drop") %>%
    mutate(label = paste0(dataset, "\n(n = ", n, ")")) %>%
    select(dataset, label) %>%
    deframe()

  if (dv$unbound == TRUE) {
    p_b <- ggplot(agg_data, aes(x = dataset, y = .data[[dv$col]], fill = dataset)) +
      scale_fill_manual(values = custom_colors) +
      labs(
        #title = paste("Distribution of", dv$label, "by Dataset"),
        x = "Dataset",
        y = dv$ylab
      ) +
      scale_x_discrete(labels = x_labels) +
      theme_minimal(base_size = 14) +
      theme(
        legend.position = "none",
        panel.grid.major.x = element_blank()
      )
    
    if (use_jitter == FALSE) {
      p_b <- p_b +
        geom_boxplot(outlier.shape = 21, color = "black", size = 0.5)
    } else {
      p_b <- p_b +
        geom_boxplot(alpha = 0.5, outlier.shape = NA, color = "black", size = 0.5) +
        geom_jitter(show.legend = FALSE, width = 0.25, shape = 21, color = "black")
    }

    print(p_b)
    if(dv$print == "unbound" || dv$print == "both") {
      ggsave(paste("plots/box_plot/", dv$col, "_unbound.pdf", sep=""),
        plot = last_plot(),
        width = 12,
        height = 6,
        units = "in",  # Inches as unit of measurement
        dpi = 300)     # Set resolution
    }
  }

  if (dv$bound == TRUE) {
    ylim <- dv$ylim
    col <- dv$col
    
    # Identify out-of-range points
    df_extreme_counts <- agg_data %>%
      filter(.data[[col]] < ylim[1] | .data[[col]] > ylim[2]) %>%
      mutate(
        arrow_y = ifelse(.data[[col]] > ylim[2], ylim[2], ylim[1]),
        arrow_dir = ifelse(.data[[col]] > ylim[2], "↑", "↓")
      ) %>%
      group_by(dataset, arrow_y, arrow_dir) %>%
      summarise(n = n(), .groups = "drop") %>%
      mutate(label = paste0("n = ", n))

    p_u <- ggplot(agg_data, aes(x = dataset, y = .data[[col]], fill = dataset))

    if (use_jitter == FALSE) {
      p_u <- p_u +
        geom_boxplot(outlier.shape = 21, color = "black", size = 0.5)
    } else {
      p_u <- p_u +
        geom_boxplot(alpha = 0.5, outlier.shape = NA, color = "black", size = 0.5) +
        geom_jitter(show.legend = FALSE, width = 0.25, shape = 21, color = "black")
    }

    p_u <- p_u +
      # Arrows for out-of-range outliers
      geom_point(data = df_extreme_counts, aes(y = arrow_y), shape = ifelse(df_extreme_counts$arrow_dir == "↑", 24, 25), size = 3) +
      #geom_text(data = df_extreme, aes(y = arrow_y, label = arrow_dir), vjust = -0.5, size = 3.5) +
      geom_text(data = df_extreme_counts,
        aes(x = dataset, y = arrow_y, label = label),
        vjust = ifelse(df_extreme_counts$arrow_dir == "↑", -1.2, 2),
        size = 3.5) +

      scale_fill_manual(values = custom_colors) +
      coord_cartesian(ylim = ylim) +
      labs(
        #title = paste("Distribution of", dv$label, "by Dataset"),
        x = "Dataset",
        y = dv$ylab
      ) +
      scale_x_discrete(labels = x_labels) +
      theme_minimal(base_size = 14) +
      theme(
        legend.position = "none",
        panel.grid.major.x = element_blank()
      )

    print(p_u)
    if(dv$print == "bound" || dv$print == "both") {
      ggsave(paste("plots/box_plot/", dv$col, "_bound.pdf", sep=""),
        plot = last_plot(),
        width = 12,
        height = 6,
        units = "in",  # Inches as unit of measurement
        dpi = 300)     # Set resolution
    }
  }
  #plots[[dv$col]] <- p  # Optional
}